# Misinfo_detection.ipynb

**Author:** Zane Zhang

**Date:** 2025/07/07 17:46

**Description:** 

Based on the fine-tuned RoBERTa-LSTM model, we perform the detection of depression misinformation. This notebook is also performed on Colab with A100.

First, we load the tuned model. Then we prepare the remaining posts for detection. Finally, we perform inference.

In [1]:
!pip install numpy==1.26.4

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoTokenizer, AutoModel
from datasets import Dataset
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from safetensors.torch import load_file
import torch
import numpy as np
import torch
import pandas as pd

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


First, we load the trained model. At this stage, we construct the class
`RobertalSIMCLassifier` to maintain the same structure. Then, we adapt the weights used
in training.

In [ ]:
# Define the custom model class again
class RobertaLSTMClassifier(nn.Module):
    def __init__(self, roberta_model_name, hidden_dim=256, lstm_layers=1, dropout=0.3, num_labels=2):
        super(RobertaLSTMClassifier, self).__init__()
        self.roberta = AutoModel.from_pretrained(roberta_model_name)
        self.lstm = nn.LSTM(input_size=self.roberta.config.hidden_size,
                            hidden_size=hidden_dim,
                            num_layers=2,
                            batch_first=True,
                            bidirectional=False)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)
        lstm_out, _ = self.lstm(sequence_output)
        pooled = lstm_out[:, -1, :]  # use the output at the last time step
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss(weight=class_weights)
            loss = loss_fct(logits, labels)
        return {'logits': logits, 'loss': loss} if loss is not None else {'logits': logits}

# Load tokenizer and model weights
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

model = RobertaLSTMClassifier('roberta-base')
# Load the model weights using safetensors
model_state_dict = load_file("/content/drive/MyDrive/DMisinfo/roberta_lstm_model/checkpoint-450/model.safetensors")
# Load the state dict into the model
model.load_state_dict(model_state_dict)
model.eval().cuda()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaLSTMClassifier(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (Layer

In [4]:
# Prepare the remaining posts
posts_df = pd.read_csv("/content/drive/MyDrive/DMisinfo/posts_en.csv")
manual_df = pd.read_excel("drive/MyDrive/DMisinfo/manual_annotation.xlsx")
manual_ids = set(manual_df['Post_id'])
unlabeled_df = posts_df[~posts_df['Post_id'].isin(manual_ids)].reset_index(drop=True)
unlabeled_df["text"] = unlabeled_df["Title_normalized"] + " " + unlabeled_df["Sentence_normalized"]

unlabeled_ds = Dataset.from_pandas(unlabeled_df)

In [5]:
# Tokenize the data
def tokenize_fn(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=tokenizer.model_max_length)

unlabeled_tokenized = unlabeled_ds.map(tokenize_fn, batched=True)
# Remove unnecessary fields
columns_to_keep = ['input_ids', 'attention_mask']
unlabeled_tokenized.set_format(type='torch', columns=columns_to_keep)

Map:   0%|          | 0/6175 [00:00<?, ? examples/s]

The next code cell is responsible for performing inference on the unlabeled data using the trained RoBERTa-LSTM model.

`data_collator = DataCollatorWithPadding(tokenizer)` initializes a data collator. This object is used in the DataLoader to pad the tokenized sequences to the same length within each batch, which is necessary for batch processing with models like RoBERTa.

`loader = DataLoader(unlabeled_tokenized, batch_size=8, collate_fn=data_collator, shuffle=False)` creates a PyTorch DataLoader. This loader will iterate over the unlabeled_tokenized dataset in batches of size 8, using the data_collator for padding. shuffle=False ensures that the order of predictions corresponds to the order of posts in the original unlabeled_df.

`model.eval()` sets the model to evaluation mode. This is crucial for disabling dropout and other training-specific layers, ensuring consistent predictions.

`with torch.no_grad()` creates a context where gradients are not calculated. This saves memory and speeds up computation during inference as we don't need to backpropagate.

`batch = {k: v.cuda() for k, v in batch.items() if k in ['input_ids', 'attention_mask']}` moves the `input_ids` and attention_mask tensors in the current batch to the GPU (`.cuda()`) if a GPU is available. This is necessary for running inference on the GPU.

`outputs = model(**batch)` passes the input tensors (`input_ids` and `attention_mask`) to the model to get the outputs.

`probs = torch.softmax(outputs['logits'], dim=1)` applies the softmax function to the model's output logits along dimension 1 (the class dimension). This converts the raw logits into probabilities, where `probs[:, 1]` represents the probability of the positive class (misinformation).

`all_probs.extend(probs[:, 1].cpu().numpy())` extracts the probabilities for the positive class from the current batch, moves them back to the CPU (`.cpu()`), converts them to a NumPy array (`.numpy()`), and extends the `all_probs` list with these probabilities.

`unlabeled_df['prob_misinfo'] = all_probs` adds a new column named 'prob_misinfo' to the `unlabeled_df` DataFrame, containing the predicted probabilities of misinformation for each post.

`unlabeled_df['label_pred'] = (unlabeled_df['prob_misinfo'] >= 0.4).astype(int)` creates another new column 'label_pred'. It assigns a predicted label of 1 (misinformation) if the predicted probability ('prob_misinfo') is greater than or equal to 0.4, and 0 otherwise. The `.astype(int)` converts the boolean result to integers.

In [6]:
# Run inference (batch-wise)
data_collator = DataCollatorWithPadding(tokenizer)
loader = DataLoader(unlabeled_tokenized, batch_size=8, collate_fn=data_collator, shuffle=False)

all_probs = []

model.eval()
with torch.no_grad():
    for batch in loader:
        batch = {k: v.cuda() for k, v in batch.items() if k in ['input_ids', 'attention_mask']}
        outputs = model(**batch)
        probs = torch.softmax(outputs['logits'], dim=1)
        all_probs.extend(probs[:, 1].cpu().numpy())

# Add predictions back
unlabeled_df['prob_misinfo'] = all_probs
unlabeled_df['label_pred'] = (unlabeled_df['prob_misinfo'] >= 0.4).astype(int)

unlabeled_df.to_csv("drive/MyDrive/DMisinfo/labeled_by_roberta_lstm.csv", index=False)